# Global Startup Ecosystem Intelligence Monitor (GSEIM)
**Deterministic, Zero-LLM Ecosystem Intelligence Platform**

This notebook runs the 17-stage GSEIM data collection, classification, deduplication, SQLite persistence, and Google Sheets export pipeline.

### CELL 1: Install Dependencies

In [ ]:
!pip install -q feedparser openpyxl gspread beautifulsoup4 lxml requests pandas python-dateutil tenacity
print('✓ Dependencies installed.')

### CELL 2: Authenticate Google & Mount Drive (Optional)

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print('✓ Authenticated with Google account.')
except Exception as e:
    print(f'Running in non-Colab or offline mode: {e}')

### CELL 3: Load Configuration & Taxonomies

In [ ]:
import sys, os
# Add repository to path
if os.path.exists('gseim'):
    sys.path.insert(0, os.path.abspath('gseim'))
else:
    sys.path.insert(0, os.path.abspath('.'))

from gseim.config.settings import DB_PATH, SPREADSHEET_TITLE, EXCEL_OUTPUT_PATH, SOURCES_SEED_PATH
from gseim.config.keywords import CATEGORY_KEYWORDS, JOB_FUNCTIONS, OPPORTUNITY_WEIGHTS, JOB_WEIGHTS
print(f'✓ Configuration loaded. Database target: {DB_PATH}')

### CELL 4: Initialize SQLite Database & Load Source Registry

In [ ]:
from gseim.database.repository import GSEIMRepository
repo = GSEIMRepository(DB_PATH)
loaded_sources = repo.load_seed_sources(SOURCES_SEED_PATH)
active_sources = repo.get_active_sources()
print(f'✓ Loaded {len(active_sources)} active sources from seed registry.')

### CELL 5: Load Historical Records

In [ ]:
existing_opps = repo.get_opportunities()
existing_jobs = repo.get_jobs()
print(f'✓ Historical state: {len(existing_opps)} opportunities, {len(existing_jobs)} jobs previously tracked.')

### CELL 6: Initialize Crawlers & Politeness Layer

In [ ]:
from gseim.collectors.rss import RSSCollector
from gseim.collectors.sitemap import SitemapCollector
from gseim.collectors.html import HTMLCollector
from gseim.collectors.ats import ATSCollector

rss_collector = RSSCollector()
sitemap_collector = SitemapCollector()
html_collector = HTMLCollector()
ats_collector = ATSCollector()
print('✓ Resilient collectors initialized with per-domain rate limiting and robots.txt checking.')

### CELL 7: Collect RSS Feeds

In [ ]:
rss_sources = [s for s in active_sources if s.get('monitoring_method') == 'RSS']
rss_raw = []
for src in rss_sources[:15]:  # Sample first 15 for demo run
    try:
        items = rss_collector.collect(src)
        rss_raw.extend(items)
    except Exception as e:
        print(f"RSS fetch error {src['source_id']}: {e}")
print(f'✓ RSS Ingestion complete: {len(rss_raw)} items collected.')

### CELL 8: Collect Sitemaps

In [ ]:
sitemap_sources = [s for s in active_sources if s.get('monitoring_method') == 'Sitemap']
sitemap_raw = []
for src in sitemap_sources[:5]:
    try:
        items = sitemap_collector.collect(src, max_items=5)
        sitemap_raw.extend(items)
    except Exception as e:
        print(f"Sitemap fetch error {src['source_id']}: {e}")
print(f'✓ Sitemap Ingestion complete: {len(sitemap_raw)} items collected.')

### CELL 9: Collect HTML & ATS Portals

In [ ]:
html_sources = [s for s in active_sources if s.get('monitoring_method') in ['HTML', 'ATS']]
html_raw = []
for src in html_sources[:10]:
    try:
        if src.get('monitoring_method') == 'ATS':
            items = ats_collector.collect(src)
        else:
            items = html_collector.collect(src)
        html_raw.extend(items)
    except Exception as e:
        print(f"HTML/ATS error {src['source_id']}: {e}")
all_raw_items = rss_raw + sitemap_raw + html_raw
print(f'✓ Total raw ecosystem documents collected: {len(all_raw_items)}')

### CELL 10: Extract Opportunities

In [ ]:
from gseim.classifiers.opportunity_classifier import classify_opportunity
candidate_opps = [classify_opportunity(item) for item in all_raw_items]
candidate_opps = [o for o in candidate_opps if o is not None]
print(f'✓ Classified {len(candidate_opps)} qualified opportunities above minimum relevance score.')

### CELL 11: Extract Jobs

In [ ]:
from gseim.classifiers.job_classifier import classify_job
candidate_jobs = [classify_job(item) for item in all_raw_items]
candidate_jobs = [j for j in candidate_jobs if j is not None]
print(f'✓ Classified {len(candidate_jobs)} qualified ecosystem jobs (Program/Incubation/Investment).')

### CELL 12: Classify & Filter Ecosystem Updates

In [ ]:
from gseim.classifiers.relevance import detect_ecosystem_update
candidate_updates = [detect_ecosystem_update(item) for item in all_raw_items]
candidate_updates = [u for u in candidate_updates if u is not None]
print(f'✓ Detected {len(candidate_updates)} ecosystem policy/fund announcements.')

### CELL 13: Deduplicate (Composite SHA256 Hashes)

In [ ]:
new_opp_count, updated_opp_count = 0, 0
for opp in candidate_opps:
    action, opp_id = repo.save_opportunity(opp)
    if action == 'NEW':
        new_opp_count += 1
    elif action == 'UPDATED':
        updated_opp_count += 1

print(f'✓ Opportunities Deduplication: {new_opp_count} NEW, {updated_opp_count} UPDATED.')

### CELL 14: Detect Updates & Save Jobs

In [ ]:
new_job_count, updated_job_count = 0, 0
for job in candidate_jobs:
    action, job_id = repo.save_job(job)
    if action == 'NEW':
        new_job_count += 1
    elif action == 'UPDATED':
        updated_job_count += 1

for upd in candidate_updates:
    repo.save_ecosystem_update(upd)

print(f'✓ Jobs Deduplication: {new_job_count} NEW, {updated_job_count} UPDATED.')

### CELL 15: Export Multi-Tab Excel Mirror & Sync Google Sheets

In [ ]:
from gseim.export.excel_mirror import export_to_excel
from gseim.export.sheets import GoogleSheetsExporter

# Export Excel Mirror
xlsx_path = export_to_excel(repo, EXCEL_OUTPUT_PATH)
print(f'✓ Multi-tab Excel workbook generated at: {xlsx_path}')

# Sync to Google Sheets if authenticated
exporter = GoogleSheetsExporter()
if exporter.is_connected():
    exporter.sync(repo, SPREADSHEET_TITLE)
    print(f'✓ Live synchronized to Google Sheets: "{SPREADSHEET_TITLE}"')
else:
    print('ℹ Google Sheets auth not active in this session. All 10 tabs saved to local Excel.')

### CELL 16: Generate & Print Execution Run Report

In [ ]:
from gseim.export.dashboard import generate_dashboard_metrics, format_run_report
metrics = generate_dashboard_metrics(repo)
print(format_run_report(metrics))

### CELL 17: Display Dashboard Metrics & Deadlines Table

In [ ]:
import pandas as pd
deadlines = repo.get_deadlines()
if deadlines:
    display(pd.DataFrame(deadlines).head(10))
else:
    print('No closing deadlines recorded in current batch.')